# Tạo split train/validation và fold CV

- [samples.csv](samples.csv) chứa ID, nhãn và mốc đồng bộ MCF lấy từ manifest trước khi chia. Có thể chạy notebook với các file trong repo.
- Chia phân tầng **80% train / 20% validation** và gán **5 fold CV** trên train.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import StratifiedKFold, train_test_split

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "papercode").is_dir())
prepare = root / "papercode" / "prepare-dataset"
split_dir = prepare / "splits"
samples = pd.read_csv(split_dir / "samples.csv", dtype={"reference_frame_index": "Int64"})

## Đơn vị chia

- **MCF** chia theo cặp (scenario, mốc đồng bộ). Sắp xếp theo scenario, mốc và camera, rồi gán một mã time_row cho cả tám ảnh cùng thời điểm.
- **UP-Fall** chia theo từng ảnh Camera 1. Giữ thứ tự ảnh trong manifest gốc khi gán time_row. Các ảnh của một subject có thể xuất hiện ở cả train và validation.

In [2]:
mcf = samples.loc[samples.sample_id.str.startswith("mcf_")].copy()
mcf[["scenario", "camera"]] = mcf.sample_id.str.extract(r"^mcf_s(\d+)_c(\d+)_f\d+$").astype(int)
mcf = mcf.sort_values(["scenario", "reference_frame_index", "camera"], ignore_index=True)
mcf["group_id"] = mcf.scenario.map("mcf_scenario_{:02d}".format)
mcf["time_row"] = mcf.groupby(["scenario", "reference_frame_index"], sort=False).ngroup()

upfall = samples.loc[samples.sample_id.str.startswith("upfall_")].reset_index(drop=True)
subject = upfall.sample_id.str.extract(r"^upfall_s(\d+)_", expand=False)
upfall["group_id"] = "upfall_subject_" + subject
upfall["time_row"] = upfall.index

## Chia 80:20 và gán fold

- `train_test_split` phân tầng theo nhãn với `test_size=0.2`, `shuffle=True` và `random_state=1`.
- `StratifiedKFold` chia phần train thành 5 fold với `shuffle=False`. Giữ thứ tự `train_indices` do bước 80:20 trả về để tái lập đúng số fold.
- `held` là vị trí trong tập train. `train_indices[held]` đưa vị trí đó về bảng đơn vị chia. Giá trị `-1` đánh dấu validation giữ riêng.

In [3]:
columns = ["sample_id", "group_id", "label", "posture", "time_row", "split", "cv_fold"]

def create_assignment(frames):
    units = frames.drop_duplicates("time_row")
    labels = units.label.to_numpy()
    train_indices, _ = train_test_split(
        np.arange(len(units)),
        test_size=0.2,
        shuffle=True,
        random_state=1,
        stratify=labels,
    )

    cv_fold = np.full(len(units), -1, dtype=int)
    cv = StratifiedKFold(n_splits=5, shuffle=False)
    for fold, (_, held) in enumerate(cv.split(train_indices, labels[train_indices])):
        cv_fold[train_indices[held]] = fold

    assignment = frames.copy()
    assignment["cv_fold"] = cv_fold[assignment.time_row.to_numpy()]
    assignment["split"] = np.where(assignment.cv_fold.ge(0), "train", "validation")
    assignment["posture"] = "P" + (assignment.label + 1).astype(str)
    return assignment[columns]

assignments = {
    "mcf": create_assignment(mcf),
    "upfall": create_assignment(upfall),
}

## Lưu CSV

Hai CSV vừa tạo được lưu dưới `prepared/splits/`.

In [4]:
output = prepare / "prepared" / "splits"
output.mkdir(parents=True, exist_ok=True)
for dataset, assignment in assignments.items():
    assignment.to_csv(output / f"{dataset}.csv", index=False)

counts = pd.DataFrame({name: table["split"].value_counts() for name, table in assignments.items()}).T
display(counts[["train", "validation"]])

fold_counts = pd.DataFrame({name: table.cv_fold.value_counts() for name, table in assignments.items()}).T
display(fold_counts.rename_axis(columns="cv_fold")[list(range(5))])

split,train,validation
mcf,21608,5408
upfall,35396,8849


cv_fold,0,1,2,3,4
mcf,4328,4320,4320,4320,4320
upfall,7080,7079,7079,7079,7079
